# Panel multinomial logit (Panel MNL)

## Model

For individual $n$, repeated choice occasion $t\in\mathcal{P}_n$, and
available alternative $j\in\mathcal{C}_{nt}$, Panel MNL uses

$$
P_{ntj}(\boldsymbol{\beta})=
\frac{a_{ntj}\exp\!\left(V_{ntj}(\boldsymbol{\beta})\right)}
{\sum_{k\in\mathcal{C}_{nt}}
a_{ntk}\exp\!\left(V_{ntk}(\boldsymbol{\beta})\right)},
\qquad
\ell(\boldsymbol{\beta})=
\sum_{n=1}^{N}\sum_{t\in\mathcal{P}_n}
\log P_{nt,y_{nt}}(\boldsymbol{\beta}).
$$

Unlike Panel MixL, Panel MNL has no individual-level random coefficient to
integrate out. The repeated-choice likelihood therefore factorizes into
ordinary MNL probabilities. The `person_id` mapping is still important for
inference because observations from the same person need not have independent
score contributions. This example reports a person-clustered sandwich
covariance:

$$
\widehat{\operatorname{Cov}}_{\mathrm{cl}}
(\widehat{\boldsymbol{\beta}})
=
\boldsymbol{H}^{+}
\left(\sum_{n=1}^{N}
\boldsymbol{s}_n\boldsymbol{s}_n^{\mathsf T}\right)
\boldsymbol{H}^{+},
\qquad
\boldsymbol{s}_n=
\sum_{t\in\mathcal{P}_n}
\frac{\partial\log P_{nt,y_{nt}}(\widehat{\boldsymbol{\beta}})}
{\partial\boldsymbol{\beta}}.
$$

The example uses TRAIN, SM, and CAR. Its systematic utilities are

$$
\begin{aligned}
V_{nt,\mathrm{TRAIN}}&=\mathrm{ASC}_{T}
+\beta_t t_{ntT}+\beta_c c_{ntT}+\gamma_T z_n,\\
V_{nt,\mathrm{SM}}&=\beta_t t_{ntS}+\beta_c c_{ntS},\\
V_{nt,\mathrm{CAR}}&=\mathrm{ASC}_{C}
+\beta_t t_{ntC}+\beta_c c_{ntC}+\gamma_C z_n.
\end{aligned}
$$

### Notation

- $N$ is the number of individuals, and $n=1,\ldots,N$ indexes one person.
  The code creates 160 `person_id` values.
- $\mathcal{P}_n$ is the set of repeated choice occasions observed for person
  $n$, and $t\in\mathcal{P}_n$ indexes one occasion. Each person has six
  occasions in this example.
- $\mathcal{C}_{nt}$ is the choice set at occasion $t$ for person $n$.
  Alternatives $j$ and $k$ index TRAIN, SM, and CAR.
- $a_{ntj}\in\{0,1\}$ is the availability indicator. An unavailable
  alternative is excluded from the probability denominator.
- $y_{nt}$ is the alternative chosen by person $n$ at occasion $t$, and
  $P_{ntj}$ is the probability of choosing alternative $j$.
- $V_{ntj}$ is systematic utility. The type-I extreme-value error underlying
  MNL is omitted from the displayed probability equation.
- $\boldsymbol{\beta}$ collects all estimated coefficients.
  $\widehat{\boldsymbol{\beta}}$ is the maximum-likelihood estimate.
- $T$, $S$, and $C$ abbreviate TRAIN, SM, and CAR. SM is the reference
  alternative, so its constant and income interaction are normalized to zero.
- $t_{ntj}$ and $c_{ntj}$ are travel time and monetary cost. They correspond
  to the `time` and `cost` variables in the code.
- $z_n$ is standardized person-level income (`income_std`). It is constant
  across the six occasions and repeated across alternatives.
- $\mathrm{ASC}_{T}$ and $\mathrm{ASC}_{C}$ are alternative-specific
  constants (`ASC_TRAIN` and `ASC_CAR`).
- $\beta_t$ and $\beta_c$ are the generic time and cost coefficients
  (`B_TIME` and `B_COST`).
- $\gamma_T$ and $\gamma_C$ are the TRAIN- and CAR-specific income effects
  (`B_INCOME_TRAIN` and `B_INCOME_CAR`).
- $\boldsymbol{s}_n$ is the score summed over all occasions for person $n$.
  $\boldsymbol{H}$ is the observed information matrix, and
  $\boldsymbol{H}^{+}$ is its pseudoinverse. Summing scores within person
  before forming their outer products permits arbitrary within-person score
  dependence in the reported covariance.

### Execution environment

Machine configuration: AMD Ryzen 9 9950X3D CPU (16 cores), 64 GB RAM, and
NVIDIA GeForce RTX 5090 GPU (32 GB VRAM), running Ubuntu 24.04.4 with
PyTorch 2.12.1 and CUDA 13.0. Install the released package with
`pip install torchdcm`, then select CPU or CUDA through the `device` argument.


In [1]:
import numpy as np
import torch

from IPython.display import HTML, display

from torchdcm import Beta, ChoiceDataset, MultinomialLogit, UtilitySpec
from torchdcm.datasets import make_swissmetro_like

# Use double precision and fixed seeds for reproducible data and estimation.
torch.set_default_dtype(torch.float64)
torch.manual_seed(31)
rng = np.random.default_rng(31)

# Use CUDA automatically when available; set device = "cpu" to force CPU.
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"TorchDCM example running on {device}")


TorchDCM example running on cuda


In [2]:
# Create six repeated choice occasions for each of 160 people.
n_people = 160
n_occasions = 6
frame = make_swissmetro_like(
    n_obs=n_people * n_occasions,
    seed=31,
)
frame["person_id"] = np.repeat(np.arange(n_people), n_occasions)

# Draw one observed income value per person and repeat it over that person's
# choice occasions. Time and cost continue to vary across occasions and modes.
person_income = rng.normal(size=n_people)
frame["income_std"] = np.repeat(person_income, n_occasions)
alternatives = np.array(["TRAIN", "SM", "CAR"])
time = frame[["time_train", "time_sm", "time_car"]].to_numpy()
cost = frame[["cost_train", "cost_sm", "cost_car"]].to_numpy()
available = frame[["avail_train", "avail_sm", "avail_car"]].to_numpy(bool)

# Generate choices from a fixed-coefficient Panel MNL data-generating process.
# The person-level income interactions make the repeated rows a nontrivial
# panel while the underlying choice probabilities remain standard MNL.
income = frame["income_std"].to_numpy()[:, None]
utility = (
    np.array([0.25, 0.0, 0.15])
    - 0.025 * time
    - 0.070 * cost
    + income * np.array([0.18, 0.0, -0.12])
)
masked_utility = np.where(available, utility, -np.inf)
probabilities = np.exp(
    masked_utility - np.max(masked_utility, axis=1, keepdims=True)
)
probabilities /= probabilities.sum(axis=1, keepdims=True)
frame["choice"] = [
    rng.choice(alternatives, p=row) for row in probabilities
]

# Convert the wide columns to TorchDCM tensors. Passing person_id records which
# choice occasions belong to the same individual.
data = ChoiceDataset.from_wide(
    frame,
    alternatives=alternatives.tolist(),
    choice="choice",
    variables={
        "time": {"TRAIN": "time_train", "SM": "time_sm", "CAR": "time_car"},
        "cost": {"TRAIN": "cost_train", "SM": "cost_sm", "CAR": "cost_car"},
        "income_std": {
            "TRAIN": "income_std",
            "SM": "income_std",
            "CAR": "income_std",
        },
    },
    availability={
        "TRAIN": "avail_train",
        "SM": "avail_sm",
        "CAR": "avail_car",
    },
    individual_id="person_id",
)

# Use generic time and cost coefficients. SM is the reference alternative;
# TRAIN and CAR receive their own constants and person-income interactions.
spec = UtilitySpec()
spec.utility(
    "TRAIN",
    Beta("ASC_TRAIN")
    + Beta("B_TIME", init=-0.02) * "time"
    + Beta("B_COST", init=-0.05) * "cost"
    + Beta("B_INCOME_TRAIN", init=0.10) * "income_std",
)
spec.utility(
    "SM",
    Beta("B_TIME", init=-0.02) * "time"
    + Beta("B_COST", init=-0.05) * "cost",
)
spec.utility(
    "CAR",
    Beta("ASC_CAR")
    + Beta("B_TIME", init=-0.02) * "time"
    + Beta("B_COST", init=-0.05) * "cost"
    + Beta("B_INCOME_CAR", init=-0.10) * "income_std",
)
print(
    f"Choice occasions: {data.n_obs}; "
    f"individuals: {data.n_individuals}; "
    f"occasions per person: {n_occasions}; "
    f"panel detected: {data.has_panel}"
)


Choice occasions: 960; individuals: 160; occasions per person: 6; panel detected: True


In [3]:
# Fit the fixed-coefficient MNL likelihood over all choice occasions.
model = MultinomialLogit(spec, device=device, max_iter=100)

# Cluster by person_id: point estimates remain MNL estimates, while the
# sandwich covariance first sums score contributions within each person.
result = model.fit(
    data,
    cov_type="cluster",
    groups="person_id",
)

# Render the complete structured estimation report in the notebook.
display(HTML(result.report().to_html()))


Model family,MultinomialLogit
Estimation objective,Maximum log likelihood
TorchDCM version,0.1.1
PyTorch version,2.12.1+cu130
Python version,3.12.13
Operating system,Linux-6.17.0-35-generic-x86_64-with-glibc2.39
Device,cuda
Tensor dtype,float64
Optimizer,torch.optim.LBFGS
Maximum iterations,100
Line search,strong_wolfe
